# WiDS Wildfire Survival — High-Score Submit Notebook

In [1]:

import os, sys, math, gc, warnings, json, random, importlib.util
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor

RUN_MODE = os.environ.get("RUN_MODE", "full").lower()
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUTPUT_PATH = os.path.join(WORK_DIR, "submission.csv")
HORIZONS = [12, 24, 48, 72]
PROB_COLS = ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]
REQ_COLS = ["event_id"] + PROB_COLS

if RUN_MODE == "fast":
    SEEDS = [11]
    N_TREES = 160
    N_BOOST = 160
    RANDOM_SEARCH = 600
else:
    SEEDS = [11, 23, 37, 51, 73, 101, 137]
    N_TREES = 360
    N_BOOST = 320
    RANDOM_SEARCH = 6500

HAS_LGB = importlib.util.find_spec("lightgbm") is not None
HAS_CAT = importlib.util.find_spec("catboost") is not None and os.environ.get("DISABLE_CAT", "0") != "1"
HAS_XGB = importlib.util.find_spec("xgboost") is not None and os.environ.get("DISABLE_XGB", "0") != "1"
if HAS_LGB:
    import lightgbm as lgb
if HAS_CAT:
    from catboost import CatBoostClassifier, CatBoostRegressor
if HAS_XGB:
    import xgboost as xgb

np.random.seed(20260421)
random.seed(20260421)


def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/widsworldwide-global-datathon-2026",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input",
        ".",
        "/mnt/data",
    ]
    need = {"train.csv", "test.csv", "sample_submission.csv"}
    for p in candidates:
        if os.path.isdir(p) and need.issubset(set(os.listdir(p))):
            return p
    for root, _, files in os.walk("/kaggle/input"):
        if need.issubset(set(files)):
            return root
    raise FileNotFoundError("train.csv, test.csv, sample_submission.csv not found")

DATA_DIR = find_data_dir()
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
print("DATA_DIR", DATA_DIR)
print("train/test", train_df.shape, test_df.shape, "packages", {"lgb": HAS_LGB, "cat": HAS_CAT, "xgb": HAS_XGB})


def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))


def enforce_monotonicity(arr):
    out = np.asarray(arr, dtype=float).copy()
    out = np.clip(out, 0.0, 1.0)
    for j in range(1, out.shape[1]):
        out[:, j] = np.maximum(out[:, j], out[:, j - 1])
    return np.clip(out, 0.0, 1.0)


def compute_c_index(time, event, risk):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    risk = np.asarray(risk, dtype=float)
    conc = 0.0
    comp = 0.0
    for i in range(len(time)):
        if event[i] != 1:
            continue
        m = time[i] < time
        m[i] = False
        if not np.any(m):
            continue
        comp += float(np.sum(m))
        rv = risk[m]
        conc += float(np.sum(risk[i] > rv)) + 0.5 * float(np.sum(risk[i] == rv))
    return conc / comp if comp > 0 else 0.5


def compute_brier(time, event, prob, horizon):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    prob = np.asarray(prob, dtype=float)
    valid = ~((event == 0) & (time < horizon))
    if valid.sum() == 0:
        return 0.25
    y = ((event == 1) & (time <= horizon)).astype(float)
    return float(np.mean((np.clip(prob[valid], 0, 1) - y[valid]) ** 2))


def compute_hybrid_score(time, event, pred):
    pred = enforce_monotonicity(pred)
    risk = 0.3 * pred[:, 1] + 0.4 * pred[:, 2] + 0.3 * pred[:, 3]
    cidx = compute_c_index(time, event, risk)
    b24 = compute_brier(time, event, pred[:, 1], 24)
    b48 = compute_brier(time, event, pred[:, 2], 48)
    b72 = compute_brier(time, event, pred[:, 3], 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * cidx + 0.7 * (1.0 - wb), cidx, wb, b24, b48, b72


def horizon_target(time, event, horizon):
    y = ((event == 1) & (time <= horizon)).astype(int)
    valid = ~((event == 0) & (time < horizon))
    return y, valid


def rank_against(values, reference):
    values = np.asarray(values, dtype=float)
    reference = np.asarray(reference, dtype=float)
    reference = reference[np.isfinite(reference)]
    if reference.size == 0:
        return np.full(values.shape, 0.5)
    reference = np.sort(reference)
    return np.searchsorted(reference, values, side="right") / reference.size


def add_features(df, ref_df):
    r = df.copy()
    ref = ref_df.copy()
    eps = 1e-9

    dist = r["dist_min_ci_0_5h"].clip(lower=1.0)
    ref_dist = ref["dist_min_ci_0_5h"].clip(lower=1.0)
    speed = r["closing_speed_m_per_h"]
    speed_pos = speed.clip(lower=0.0)
    radial = r["radial_growth_rate_m_per_h"]
    radial_pos = radial.clip(lower=0.0)
    align = r["alignment_abs"].clip(0, 1)
    area = r["area_first_ha"].clip(lower=0.0)
    growth_abs = r["area_growth_abs_0_5h"]
    growth_pos = growth_abs.clip(lower=0.0)
    fire_radius = np.sqrt(area * 10000.0 / np.pi)
    eff_close = speed_pos + radial_pos
    eff_align = eff_close * align

    r["dist_km"] = dist / 1000.0
    r["log_dist"] = np.log1p(dist)
    r["sqrt_dist"] = np.sqrt(dist)
    r["inv_dist_km"] = 1.0 / (dist / 1000.0 + 0.05)
    r["inv_dist_km2"] = r["inv_dist_km"] ** 2
    r["d_minus_5km"] = dist - 5000.0
    r["abs_d_minus_5km"] = np.abs(dist - 5000.0)
    r["inside_5km"] = (dist < 5000.0).astype(float)
    for thr in [250, 500, 750, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000, 5500, 6000, 8000, 10000, 15000, 20000, 50000]:
        r[f"dist_lt_{thr}"] = (dist < thr).astype(float)

    r["fire_radius_m"] = fire_radius
    r["fire_radius_km"] = fire_radius / 1000.0
    r["dist_minus_radius"] = dist - fire_radius
    r["radius_to_dist"] = fire_radius / (dist + 1.0)
    r["area_to_dist"] = area / (dist / 1000.0 + 0.1)
    r["log_area_minus_log_dist"] = np.log1p(area) - np.log1p(dist)

    r["speed_pos"] = speed_pos
    r["speed_neg"] = (-speed).clip(lower=0.0)
    r["radial_pos"] = radial_pos
    r["eff_close"] = eff_close
    r["eff_align"] = eff_align
    r["speed_align"] = speed * align
    r["speed_pos_align"] = speed_pos * align
    r["growth_pos"] = growth_pos
    r["growth_per_area"] = growth_abs / (area + 1.0)
    r["growth_rate_per_area"] = r["area_growth_rate_ha_per_h"] / (area + 1.0)
    r["radius_growth_to_radius"] = r["radial_growth_m"] / (fire_radius + 1.0)

    for c in [
        "area_first_ha", "area_growth_abs_0_5h", "area_growth_rate_ha_per_h",
        "radial_growth_m", "radial_growth_rate_m_per_h", "centroid_displacement_m",
        "centroid_speed_m_per_h", "closing_speed_m_per_h", "closing_speed_abs_m_per_h",
        "along_track_speed", "cross_track_component", "dist_change_ci_0_5h", "dist_slope_ci_0_5h",
        "dist_accel_m_per_h2"
    ]:
        if c in r.columns:
            x = r[c].astype(float)
            r[f"abs_{c}"] = np.abs(x)
            r[f"signed_log_{c}"] = np.sign(x) * np.log1p(np.abs(x))

    r["projected_advance_pos"] = r["projected_advance_m"].clip(lower=0.0)
    r["closing_distance_pos"] = (-r["dist_change_ci_0_5h"]).clip(lower=0.0)
    r["slope_toward"] = (-r["dist_slope_ci_0_5h"]).clip(lower=0.0)
    r["accel_toward"] = (-r["dist_accel_m_per_h2"]).clip(lower=0.0)
    r["advance_to_dist"] = r["projected_advance_pos"] / (dist + 1.0)
    r["closing5_to_dist"] = r["closing_distance_pos"] / (dist + 1.0)
    r["motion_conf"] = r["dist_fit_r2_0_5h"] * (r["num_perimeters_0_5h"] > 1).astype(float)
    r["threat"] = align * (eff_close + 1.0) / np.log1p(dist)
    r["threat2"] = r["threat"] ** 2
    r["area_x_close"] = np.log1p(area) * speed_pos
    r["area_x_align"] = np.log1p(area) * align
    r["dist_x_lowres"] = r["dist_km"] * r["low_temporal_resolution_0_5h"]
    r["align_x_lowres"] = align * r["low_temporal_resolution_0_5h"]

    eta_speed = np.where(speed_pos > 0.01, dist / np.maximum(speed_pos, eps), 99999.0)
    eta_eff = np.where(eff_close > 0.01, dist / np.maximum(eff_close, eps), 99999.0)
    eta_align = np.where(eff_align > 0.01, dist / np.maximum(eff_align, eps), 99999.0)
    for nm, eta in [("eta_speed", eta_speed), ("eta_eff", eta_eff), ("eta_align", eta_align)]:
        eta = np.clip(eta, 0.0, 99999.0)
        r[nm] = eta
        r[f"log_{nm}"] = np.log1p(eta)
        for H in [12, 24, 48, 72]:
            r[f"{nm}_le_{H}"] = (eta <= H).astype(float)
            r[f"{nm}_margin_{H}"] = np.clip((H - eta) / H, -10, 10)

    r["hour_sin"] = np.sin(2 * np.pi * r["event_start_hour"] / 24.0)
    r["hour_cos"] = np.cos(2 * np.pi * r["event_start_hour"] / 24.0)
    r["month_sin"] = np.sin(2 * np.pi * (r["event_start_month"] - 1) / 12.0)
    r["month_cos"] = np.cos(2 * np.pi * (r["event_start_month"] - 1) / 12.0)
    r["dow_sin"] = np.sin(2 * np.pi * r["event_start_dayofweek"] / 7.0)
    r["dow_cos"] = np.cos(2 * np.pi * r["event_start_dayofweek"] / 7.0)
    for m in range(1, 13):
        r[f"month_{m}"] = (r["event_start_month"] == m).astype(float)
    for b, (lo, hi) in enumerate([(0, 6), (6, 12), (12, 18), (18, 24)]):
        r[f"hour_bin_{b}"] = ((r["event_start_hour"] >= lo) & (r["event_start_hour"] < hi)).astype(float)

    ref_eff = ref["closing_speed_m_per_h"].clip(lower=0.0) + ref["radial_growth_rate_m_per_h"].clip(lower=0.0)
    ref_area = ref["area_first_ha"].clip(lower=0.0)
    ref_threat = ref["alignment_abs"].clip(0, 1) * (ref_eff + 1.0) / np.log1p(ref_dist)
    for nm, val, ref_val in [
        ("rank_dist", dist, ref_dist),
        ("rank_eff", eff_close, ref_eff),
        ("rank_area", area, ref_area),
        ("rank_threat", r["threat"], ref_threat),
    ]:
        r[nm] = rank_against(np.asarray(val, dtype=float), np.asarray(ref_val, dtype=float))

    r = r.drop(columns=["event_id", "time_to_hit_hours", "event"], errors="ignore")
    r = r.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return r

X_train = add_features(train_df, train_df)
X_test = add_features(test_df, train_df)
feature_cols = list(X_train.columns)
X_test = X_test.reindex(columns=feature_cols, fill_value=0.0)
print("features", len(feature_cols))

time_values = train_df["time_to_hit_hours"].to_numpy(dtype=float)
event_values = train_df["event"].to_numpy(dtype=int)
dist_train = train_df["dist_min_ci_0_5h"].to_numpy(dtype=float)
dist_test = test_df["dist_min_ci_0_5h"].to_numpy(dtype=float)
near_train = dist_train < 5000.0
near_test = dist_test < 5000.0


def make_classifier(name, seed, variant=0):
    if name == "et":
        leaves = [2, 3, 4, 5][variant % 4]
        mf = [0.55, 0.70, 0.85, None][variant % 4]
        return ExtraTreesClassifier(
            n_estimators=N_TREES, min_samples_leaf=leaves, max_features=mf,
            random_state=seed + 101 * variant, n_jobs=1, class_weight="balanced",
            bootstrap=False
        )
    if name == "rf":
        leaves = [3, 4, 5][variant % 3]
        mf = [0.55, 0.75, None][variant % 3]
        return RandomForestClassifier(
            n_estimators=max(180, N_TREES // 2), min_samples_leaf=leaves, max_features=mf,
            random_state=seed + 101 * variant, n_jobs=1, class_weight="balanced_subsample",
            bootstrap=True
        )
    if name == "lr":
        C = [0.08, 0.15, 0.30][variant % 3]
        return make_pipeline(RobustScaler(), LogisticRegression(C=C, penalty="l2", solver="liblinear", class_weight="balanced", max_iter=2000, random_state=seed))
    if name == "lgb" and HAS_LGB:
        return lgb.LGBMClassifier(
            n_estimators=N_BOOST, learning_rate=[0.025, 0.035, 0.045][variant % 3],
            num_leaves=[5, 7, 9][variant % 3], max_depth=[2, 3, 3][variant % 3],
            min_child_samples=[5, 8, 12][variant % 3], subsample=[0.82, 0.90, 0.75][variant % 3],
            colsample_bytree=[0.72, 0.82, 0.92][variant % 3], reg_alpha=[0.10, 0.25, 0.00][variant % 3],
            reg_lambda=[2.5, 4.0, 1.5][variant % 3], objective="binary",
            random_state=seed + 101 * variant, verbosity=-1, n_jobs=1
        )
    if name == "cat" and HAS_CAT:
        return CatBoostClassifier(
            iterations=N_BOOST, learning_rate=[0.025, 0.035][variant % 2], depth=[3, 4][variant % 2],
            l2_leaf_reg=[8.0, 12.0][variant % 2], loss_function="Logloss", eval_metric="Logloss",
            random_seed=seed + 101 * variant, verbose=False, allow_writing_files=False,
            bootstrap_type="Bayesian", bagging_temperature=[0.4, 0.8][variant % 2], thread_count=1
        )
    if name == "xgb" and HAS_XGB:
        return xgb.XGBClassifier(
            n_estimators=N_BOOST, learning_rate=[0.025, 0.035][variant % 2], max_depth=[2, 3][variant % 2],
            min_child_weight=[2.5, 4.0][variant % 2], subsample=[0.82, 0.92][variant % 2],
            colsample_bytree=[0.78, 0.90][variant % 2], reg_alpha=[0.05, 0.15][variant % 2],
            reg_lambda=[3.0, 5.0][variant % 2], objective="binary:logistic", eval_metric="logloss",
            random_state=seed + 101 * variant, n_jobs=1, tree_method="hist"
        )
    raise ValueError(name)


def predict_proba_safe(model, X):
    p = model.predict_proba(X)
    if p.ndim == 2 and p.shape[1] > 1:
        return p[:, 1]
    return np.asarray(p).reshape(-1)


def fit_horizon_family(name, variant=0, seeds=SEEDS):
    oof = np.zeros((len(train_df), 4), dtype=float)
    pred_test = np.zeros((len(test_df), 4), dtype=float)
    for hi, H in enumerate(HORIZONS):
        if H == 72:
            oof[:, hi] = 1.0
            pred_test[:, hi] = 1.0
            continue
        y, valid = horizon_target(time_values, event_values, H)
        if valid.sum() < 20 or len(np.unique(y[valid])) < 2:
            base = float(np.mean(y[valid])) if valid.sum() else 0.0
            oof[:, hi] = base
            pred_test[:, hi] = base
            continue
        pred = np.zeros(len(train_df), dtype=float)
        test_acc = np.zeros(len(test_df), dtype=float)
        valid_idx = np.where(valid)[0]
        strat_valid = y[valid].astype(str) + "_" + (dist_train[valid] < 5000.0).astype(int).astype(str)
        for seed in seeds:
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
            for tr_rel, va_rel in cv.split(X_train.iloc[valid_idx], strat_valid):
                tr_idx = valid_idx[tr_rel]
                va_idx = valid_idx[va_rel]
                model = make_classifier(name, seed, variant)
                try:
                    model.fit(X_train.iloc[tr_idx], y[tr_idx])
                    pred[va_idx] += predict_proba_safe(model, X_train.iloc[va_idx]) / len(seeds)
                finally:
                    del model
                    gc.collect()
            model = make_classifier(name, seed, variant)
            try:
                model.fit(X_train.iloc[valid_idx], y[valid_idx])
                test_acc += predict_proba_safe(model, X_test) / len(seeds)
                if (~valid).any():
                    pred[~valid] += predict_proba_safe(model, X_train.iloc[~valid]) / len(seeds)
            finally:
                del model
                gc.collect()
        oof[:, hi] = pred
        pred_test[:, hi] = test_acc
    oof = enforce_monotonicity(oof)
    pred_test = enforce_monotonicity(pred_test)
    oof[:, 3] = 1.0
    pred_test[:, 3] = 1.0
    return oof, pred_test


def make_regressor(name, seed, variant=0):
    if name == "etr":
        return ExtraTreesRegressor(
            n_estimators=N_TREES, min_samples_leaf=[2, 3, 4][variant % 3],
            max_features=[0.60, 0.75, 0.90][variant % 3], random_state=seed + 101 * variant,
            n_jobs=1, bootstrap=False
        )
    if name == "rfr":
        return RandomForestRegressor(
            n_estimators=max(180, N_TREES // 2), min_samples_leaf=[3, 4][variant % 2],
            max_features=[0.65, 0.85][variant % 2], random_state=seed + 101 * variant,
            n_jobs=1, bootstrap=True
        )
    if name == "ridge":
        return make_pipeline(RobustScaler(), Ridge(alpha=[8.0, 18.0, 35.0][variant % 3]))
    if name == "lgb" and HAS_LGB:
        return lgb.LGBMRegressor(
            n_estimators=N_BOOST, learning_rate=[0.025, 0.035, 0.045][variant % 3],
            num_leaves=[5, 7, 9][variant % 3], max_depth=[2, 3, 3][variant % 3],
            min_child_samples=[5, 8, 12][variant % 3], subsample=[0.82, 0.90, 0.75][variant % 3],
            colsample_bytree=[0.72, 0.82, 0.92][variant % 3], reg_lambda=[2.0, 4.0, 1.2][variant % 3],
            objective="regression", random_state=seed + 101 * variant, verbosity=-1, n_jobs=1
        )
    if name == "cat" and HAS_CAT:
        return CatBoostRegressor(
            iterations=N_BOOST, learning_rate=[0.025, 0.035][variant % 2], depth=[3, 4][variant % 2],
            l2_leaf_reg=[8.0, 12.0][variant % 2], loss_function="RMSE", random_seed=seed + 101 * variant,
            verbose=False, allow_writing_files=False, thread_count=1
        )
    if name == "xgb" and HAS_XGB:
        return xgb.XGBRegressor(
            n_estimators=N_BOOST, learning_rate=[0.025, 0.035][variant % 2], max_depth=[2, 3][variant % 2],
            min_child_weight=[2.5, 4.0][variant % 2], subsample=[0.82, 0.92][variant % 2],
            colsample_bytree=[0.78, 0.90][variant % 2], reg_alpha=[0.05, 0.15][variant % 2],
            reg_lambda=[3.0, 5.0][variant % 2], random_state=seed + 101 * variant,
            n_jobs=1, tree_method="hist", objective="reg:squarederror"
        )
    raise ValueError(name)


def risk_target(mode):
    t = time_values
    e = event_values
    if mode == "risk01":
        return np.where(e == 1, (72.0 - t) / 72.0, -t / 200.0)
    if mode == "neglog":
        return np.where(e == 1, -np.log1p(t), -np.log1p(90.0 + t))
    if mode == "logistic_time":
        return np.where(e == 1, 1.0 / (1.0 + t / 12.0), -0.02 * t / 72.0)
    if mode == "ordinal":
        return np.where(e == 1, 1.0 - t / 80.0, -0.10 - t / 250.0)
    raise ValueError(mode)


def fit_risk_family(name, mode="risk01", variant=0, seeds=SEEDS):
    y = risk_target(mode)
    oof = np.zeros(len(train_df), dtype=float)
    pred_test = np.zeros(len(test_df), dtype=float)
    strat = event_values.astype(str)
    for seed in seeds:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(X_train, strat):
            model = make_regressor(name, seed, variant)
            try:
                model.fit(X_train.iloc[tr_idx], y[tr_idx])
                oof[va_idx] += np.asarray(model.predict(X_train.iloc[va_idx]), dtype=float) / len(seeds)
            finally:
                del model
                gc.collect()
        model = make_regressor(name, seed, variant)
        try:
            model.fit(X_train, y)
            pred_test += np.asarray(model.predict(X_test), dtype=float) / len(seeds)
        finally:
            del model
            gc.collect()
    return oof, pred_test


def physical_candidate(df):
    d = df["dist_min_ci_0_5h"].to_numpy(dtype=float)
    per = df["num_perimeters_0_5h"].to_numpy(dtype=float)
    dt = df["dt_first_last_0_5h"].to_numpy(dtype=float)
    low = df["low_temporal_resolution_0_5h"].to_numpy(dtype=float)
    align = df["alignment_abs"].to_numpy(dtype=float)
    growth = np.log1p(np.clip(df["area_growth_abs_0_5h"].to_numpy(dtype=float), 0, None))
    area = np.log1p(np.clip(df["area_first_ha"].to_numpy(dtype=float), 0, None))
    speed = np.clip(df["closing_speed_m_per_h"].to_numpy(dtype=float), 0, None)
    radial = np.clip(df["radial_growth_rate_m_per_h"].to_numpy(dtype=float), 0, None)
    gate = sigmoid((5000.0 - d) / 65.0)
    s = 0.90 * (per - 1.0) + 0.85 * np.minimum(dt, 5.0) / 5.0 + 1.15 * align + 0.28 * growth + 0.08 * area + 0.004 * (speed + radial) - 1.35 * low
    p12 = gate * sigmoid(1.40 * (s - 0.02))
    p24 = gate * sigmoid(1.30 * (s + 1.18))
    p48 = gate * sigmoid(1.20 * (s + 2.15))
    floor = np.exp(-np.maximum(d - 5000.0, 0.0) / 18000.0)
    p12 = np.maximum(p12, 0.002 * floor)
    p24 = np.maximum(p24, 0.003 * floor)
    p48 = np.maximum(p48, 0.004 * floor)
    out = np.column_stack([p12, p24, p48, np.ones(len(df))])
    out = enforce_monotonicity(out)
    out[:, 3] = 1.0
    return out


def apply_distance_gate(pred, dist, floor_scale=1.0, sharp=65.0, cap_far=True):
    out = enforce_monotonicity(pred)
    dist = np.asarray(dist, dtype=float)
    gate = sigmoid((5000.0 - dist) / sharp)
    far_floor = np.column_stack([
        0.0015 * np.exp(-np.maximum(dist - 5000.0, 0.0) / 15000.0),
        0.0025 * np.exp(-np.maximum(dist - 5000.0, 0.0) / 15000.0),
        0.0035 * np.exp(-np.maximum(dist - 5000.0, 0.0) / 15000.0),
    ]) * floor_scale
    out[:, :3] = gate[:, None] * out[:, :3] + (1.0 - gate[:, None]) * far_floor
    if cap_far:
        cap = far_floor + np.array([0.004, 0.006, 0.009]) * floor_scale
        far = dist >= 5000.0
        out[far, :3] = np.minimum(out[far, :3], cap[far])
    out[:, 3] = 1.0
    return enforce_monotonicity(out)


def refine_with_rank(base, risk, alpha=0.35, eps=0.035, temp=8.0):
    base = enforce_monotonicity(base)
    q = rank_against(risk, risk)
    sharp = sigmoid(temp * (base[:, :3] - 0.50))
    comp = (1.0 - eps) * sharp + eps * q[:, None]
    out = base.copy()
    out[:, :3] = (1.0 - alpha) * base[:, :3] + alpha * comp
    out[:, 3] = 1.0
    return enforce_monotonicity(out)


def refine_test_with_rank(base_train, base_test, risk_train, risk_test, alpha=0.35, eps=0.035, temp=8.0):
    base_test = enforce_monotonicity(base_test)
    q = rank_against(risk_test, risk_train)
    sharp = sigmoid(temp * (base_test[:, :3] - 0.50))
    comp = (1.0 - eps) * sharp + eps * q[:, None]
    out = base_test.copy()
    out[:, :3] = (1.0 - alpha) * base_test[:, :3] + alpha * comp
    out[:, 3] = 1.0
    return enforce_monotonicity(out)

# Base horizon models.
base_model_specs = [("et", 0), ("et", 1), ("et", 2), ("lr", 0), ("lr", 1)]
if HAS_LGB:
    base_model_specs += [("lgb", 0), ("lgb", 1), ("lgb", 2)]
if HAS_CAT:
    base_model_specs += [("cat", 0), ("cat", 1)]
if HAS_XGB:
    base_model_specs += [("xgb", 0), ("xgb", 1)]

base_predictions = []
for name, variant in base_model_specs:
    try:
        print("fit horizon", name, variant)
        o, t = fit_horizon_family(name, variant)
        sc = compute_hybrid_score(time_values, event_values, o)
        base_predictions.append({"name": f"{name}{variant}", "oof": o, "test": t, "score": sc[0], "cidx": sc[1], "wb": sc[2]})
        print(f"  {name}{variant}: score={sc[0]:.6f} c={sc[1]:.6f} wb={sc[2]:.6f}")
    except Exception as ex:
        print("  skipped", name, variant, repr(ex))

# Risk models for ordering within the Brier-safe probability bands.
risk_specs = [("etr", "risk01", 0), ("etr", "neglog", 0), ("etr", "logistic_time", 1), ("ridge", "risk01", 0), ("ridge", "neglog", 1)]
if HAS_LGB:
    risk_specs += [("lgb", "risk01", 0), ("lgb", "neglog", 1)]
if HAS_CAT:
    risk_specs += [("cat", "risk01", 0)]
if HAS_XGB:
    risk_specs += [("xgb", "risk01", 0)]

risk_predictions = []
for name, mode, variant in risk_specs:
    try:
        print("fit risk", name, mode, variant)
        ro, rt = fit_risk_family(name, mode, variant)
        c = compute_c_index(time_values, event_values, ro)
        risk_predictions.append({"name": f"{name}_{mode}_{variant}", "oof": ro, "test": rt, "cidx": c})
        print(f"  risk {name}_{mode}_{variant}: c={c:.6f}")
    except Exception as ex:
        print("  skipped risk", name, mode, variant, repr(ex))

risk_predictions = sorted(risk_predictions, key=lambda z: z["cidx"], reverse=True)
base_predictions = sorted(base_predictions, key=lambda z: z["score"], reverse=True)

# Candidate pool.
candidates = []

def add_candidate(name, oof, test):
    oof = enforce_monotonicity(oof)
    test = enforce_monotonicity(test)
    oof[:, 3] = 1.0
    test[:, 3] = 1.0
    if not np.isfinite(oof).all() or not np.isfinite(test).all():
        return
    sc = compute_hybrid_score(time_values, event_values, oof)
    candidates.append({"name": name, "oof": oof, "test": test, "score": sc[0], "cidx": sc[1], "wb": sc[2]})

phys_oof = physical_candidate(train_df)
phys_test = physical_candidate(test_df)
add_candidate("physics", phys_oof, phys_test)
add_candidate("physics_gate", apply_distance_gate(phys_oof, dist_train, 0.7, 55.0), apply_distance_gate(phys_test, dist_test, 0.7, 55.0))

for bp in base_predictions:
    add_candidate(bp["name"], bp["oof"], bp["test"])
    add_candidate(bp["name"] + "_gate55", apply_distance_gate(bp["oof"], dist_train, 0.75, 55.0), apply_distance_gate(bp["test"], dist_test, 0.75, 55.0))
    add_candidate(bp["name"] + "_gate90", apply_distance_gate(bp["oof"], dist_train, 0.95, 90.0), apply_distance_gate(bp["test"], dist_test, 0.95, 90.0))

for bp in base_predictions[:max(5, min(8, len(base_predictions)) )]:
    for rp in risk_predictions[:max(3, min(5, len(risk_predictions)) )]:
        for alpha, eps, temp in [(0.22, 0.020, 7.0), (0.36, 0.035, 8.5), (0.50, 0.050, 10.0)]:
            ro = refine_with_rank(bp["oof"], rp["oof"], alpha=alpha, eps=eps, temp=temp)
            rt = refine_test_with_rank(bp["oof"], bp["test"], rp["oof"], rp["test"], alpha=alpha, eps=eps, temp=temp)
            add_candidate(f"{bp['name']}__{rp['name']}__rank_a{alpha}", ro, rt)
            add_candidate(f"{bp['name']}__{rp['name']}__rank_gate_a{alpha}", apply_distance_gate(ro, dist_train, 0.80, 60.0), apply_distance_gate(rt, dist_test, 0.80, 60.0))

# Deduplicate and select robust high-OOF candidates.
unique = []
for cand in sorted(candidates, key=lambda z: z["score"], reverse=True):
    duplicate = False
    for kept in unique:
        mad = float(np.mean(np.abs(cand["oof"][:, :3] - kept["oof"][:, :3])))
        if mad < 1e-5:
            duplicate = True
            break
    if not duplicate:
        unique.append(cand)
candidates = unique
candidates = sorted(candidates, key=lambda z: z["score"], reverse=True)

max_keep = 22 if RUN_MODE != "fast" else 10
selected = candidates[:max_keep]
# Ensure the best raw physics and best raw horizon model survive even if gated/refined dominate.
for must in ["physics", base_predictions[0]["name"] if base_predictions else None]:
    if must and all(c["name"] != must for c in selected):
        for c in candidates:
            if c["name"] == must:
                selected.append(c)
                break

print("candidates selected", len(selected))
for c in selected[:20]:
    print(f"  {c['score']:.6f} c={c['cidx']:.6f} wb={c['wb']:.6f}  {c['name']}")

P = np.stack([c["oof"] for c in selected], axis=0)
PT = np.stack([c["test"] for c in selected], axis=0)
score_vec = np.array([c["score"] for c in selected], dtype=float)
K = len(selected)

# Score-weighted prior, capped to prevent one overfit candidate from dominating.
tau = 0.0028
raw = np.exp((score_vec - score_vec.max()) / tau)
prior = raw / raw.sum()
cap = 0.30
for _ in range(12):
    over = prior > cap
    if not np.any(over):
        break
    excess = prior[over].sum() - cap * over.sum()
    prior[over] = cap
    if (~over).sum() > 0:
        prior[~over] += excess * prior[~over] / prior[~over].sum()
prior = prior / prior.sum()


def blend_from_weights(w):
    out = np.tensordot(w, P, axes=(0, 0))
    out = enforce_monotonicity(out)
    out[:, 3] = 1.0
    return out


def test_blend_from_weights(w):
    out = np.tensordot(w, PT, axes=(0, 0))
    out = enforce_monotonicity(out)
    out[:, 3] = 1.0
    return out


def obj_soft(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - z.max())
    w = e / e.sum()
    pred = blend_from_weights(w)
    sc = compute_hybrid_score(time_values, event_values, pred)[0]
    reg = 0.010 * float(np.sum(((w - prior) ** 2) / (prior + 0.015)))
    entropy_guard = 0.0015 * float(np.sum(w * np.log(np.maximum(w, 1e-12))))
    return -sc + reg + entropy_guard

best_w = prior.copy()
best_score = compute_hybrid_score(time_values, event_values, blend_from_weights(best_w))[0]
rng = np.random.default_rng(20260421)
alpha_prior = 1.0 + 120.0 * prior
for _ in range(RANDOM_SEARCH):
    if rng.random() < 0.75:
        w = rng.dirichlet(alpha_prior)
    else:
        w = rng.dirichlet(np.ones(K))
    # shrink random draws toward prior for lower variance.
    shrink = rng.uniform(0.35, 0.85)
    w = shrink * w + (1.0 - shrink) * prior
    pred = blend_from_weights(w)
    sc = compute_hybrid_score(time_values, event_values, pred)[0] - 0.006 * float(np.sum(((w - prior) ** 2) / (prior + 0.015)))
    if sc > best_score:
        best_score = sc
        best_w = w.copy()

# A random-search optimizer is used instead of a brittle high-dimensional local optimizer.
# This is deliberately regularized toward the OOF-score prior.
opt_w = best_w

# Conservative blend of optimized and prior weights to reduce leaderboard-probe style overfit.
final_w = 0.72 * opt_w + 0.28 * prior
final_w = final_w / final_w.sum()
final_oof = blend_from_weights(final_w)
final_test = test_blend_from_weights(final_w)

# Choose the final distance-gating variant by OOF, then apply the same transformation to test.
post_variants = []
post_variants.append(("blend", final_oof, final_test))
for floor_scale, sharp in [(0.55, 45.0), (0.70, 55.0), (0.85, 70.0), (1.00, 90.0)]:
    post_variants.append((f"gate_f{floor_scale}_s{sharp}", apply_distance_gate(final_oof, dist_train, floor_scale, sharp), apply_distance_gate(final_test, dist_test, floor_scale, sharp)))
post_scored = [(nm, compute_hybrid_score(time_values, event_values, oo)[0], oo, tt) for nm, oo, tt in post_variants]
post_scored = sorted(post_scored, key=lambda x: x[1], reverse=True)
post_name, post_score, final_oof, final_test = post_scored[0]

# Final safety calibration: keep 72h at exactly 1 and enforce monotonicity/schema.
final_test = enforce_monotonicity(final_test)
final_test[:, 3] = 1.0
final_test = np.clip(final_test, 0.0, 1.0)
final_test = enforce_monotonicity(final_test)
final_test[:, 3] = 1.0

final_oof_score = compute_hybrid_score(time_values, event_values, final_oof)
print("final postprocess", post_name)
print(f"OOF hybrid={final_oof_score[0]:.6f} c={final_oof_score[1]:.6f} wb={final_oof_score[2]:.6f} b24={final_oof_score[3]:.6f} b48={final_oof_score[4]:.6f} b72={final_oof_score[5]:.6f}")
print("weights")
for c, w in sorted(zip(selected, final_w), key=lambda z: -z[1])[:20]:
    print(f"  {w:.5f}  {c['score']:.6f}  {c['name']}")

pred_df = pd.DataFrame({
    "event_id": test_df["event_id"].values,
    "prob_12h": final_test[:, 0],
    "prob_24h": final_test[:, 1],
    "prob_48h": final_test[:, 2],
    "prob_72h": final_test[:, 3],
})
submission = sample_sub[["event_id"]].merge(pred_df, on="event_id", how="left")

# Strict validation.
assert list(submission.columns) == REQ_COLS
assert len(submission) == len(sample_sub)
assert submission["event_id"].is_unique
assert set(submission["event_id"]) == set(sample_sub["event_id"])
assert not submission[PROB_COLS].isna().any().any()
assert np.all((submission[PROB_COLS].to_numpy() >= 0.0) & (submission[PROB_COLS].to_numpy() <= 1.0))
vals = submission[PROB_COLS].to_numpy(dtype=float)
assert np.all(vals[:, 0] <= vals[:, 1] + 1e-12)
assert np.all(vals[:, 1] <= vals[:, 2] + 1e-12)
assert np.all(vals[:, 2] <= vals[:, 3] + 1e-12)

submission.to_csv(OUTPUT_PATH, index=False)
with open(os.path.join(WORK_DIR, "model_summary.json"), "w") as f:
    json.dump({
        "run_mode": RUN_MODE,
        "data_dir": DATA_DIR,
        "n_features": len(feature_cols),
        "postprocess": post_name,
        "oof_score": [float(x) for x in final_oof_score],
        "selected": [{"name": c["name"], "score": float(c["score"]), "cidx": float(c["cidx"]), "wb": float(c["wb"]), "weight": float(w)} for c, w in zip(selected, final_w)],
    }, f, indent=2)
print("saved", OUTPUT_PATH)
print(submission.head())


DATA_DIR /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
train/test (221, 37) (95, 35) packages {'lgb': True, 'cat': True, 'xgb': True}
features 175
fit horizon et 0
  et0: score=0.970159 c=0.931931 wb=0.013457
fit horizon et 1
  et1: score=0.971802 c=0.935492 wb=0.012637
fit horizon et 2
  et2: score=0.971648 c=0.935078 wb=0.012679
fit horizon lr 0
  lr0: score=0.867359 c=0.901706 wb=0.147361
fit horizon lr 1
  lr1: score=0.864321 c=0.898104 wb=0.150157
fit horizon lgb 0
  lgb0: score=0.969644 c=0.931186 wb=0.013874
fit horizon lgb 1
  lgb1: score=0.969846 c=0.930026 wb=0.013088
fit horizon lgb 2
  lgb2: score=0.970019 c=0.930441 wb=0.013018
fit horizon cat 0
  cat0: score=0.966004 c=0.926714 wb=0.017157
fit horizon cat 1
  cat1: score=0.964986 c=0.921746 wb=0.016483
fit horizon xgb 0
  xgb0: score=0.966687 c=0.933256 wb=0.018986
fit horizon xgb 1
  xgb1: score=0.965360 c=0.930937 wb=0.019887
fit risk etr risk01 0
  risk etr_risk01_0: c=0.940792
fit risk etr neglog 0
  risk et